# Day 2 — Control Flow and Functions
## Master's Course: Quantitative Skills for Biology

Yesterday we stored data in variables. Today we learn to **make decisions**, **repeat operations**, and **package logic into reusable functions**.

By the end of this session you will be able to:
- Use `if/elif/else` to branch based on conditions
- Use `for` and `while` loops to repeat operations
- Write your own functions to avoid repeating code
- Apply these tools to analyse DNA sequences and expression data

---
## 1. `if / elif / else` — making decisions

Biological data analysis is full of decisions: *Is this gene significantly expressed? Is this p-value below threshold? Is this fold-change large enough to matter?*

Python's `if` statement lets us encode these decisions.

In [ ]:
# Classify a p-value
p_value = 0.0008

if p_value < 0.001:
    print("Highly significant (p < 0.001)")
elif p_value < 0.05:
    print("Significant (p < 0.05)")
else:
    print("Not significant")

In [ ]:
# Classify expression fold-change (log2 scale)
log2fc = 2.8   # try changing this value

if log2fc > 2:
    label = "strongly upregulated"
elif log2fc > 1:
    label = "upregulated"
elif log2fc < -2:
    label = "strongly downregulated"
elif log2fc < -1:
    label = "downregulated"
else:
    label = "unchanged"

print(f"log2FC = {log2fc:.2f} → {label}")

In [ ]:
# Combining conditions: differentially expressed genes require BOTH
# a significant p-value AND a meaningful fold-change
log2fc = 1.5
p_value = 0.03

if abs(log2fc) > 1 and p_value < 0.05:
    print("Differentially expressed!")
else:
    print("Not differentially expressed.")

---
## 2. `for` loops — repeating operations

A `for` loop runs the same block of code for each item in a collection. This is essential when processing many genes, samples, or sequences.

In [ ]:
# Iterate over a list of gene names
genes = ["BRCA1", "TP53", "EGFR", "MYC", "PTEN"]

for gene in genes:
    print(f"Processing gene: {gene}")

In [ ]:
# Iterate over characters in a DNA string to count G and C
seq = "ATGCGATCGATCGAAC"

gc_count = 0
for base in seq:
    if base == "G" or base == "C":
        gc_count += 1   # shorthand for gc_count = gc_count + 1

print(f"Sequence: {seq}")
print(f"GC count: {gc_count}")
print(f"GC content: {gc_count / len(seq) * 100:.1f}%")

In [ ]:
# range() — generate a sequence of numbers
# Useful for iterating by index
samples = ["ctrl_1", "ctrl_2", "ctrl_3", "treat_1", "treat_2", "treat_3"]

for i in range(len(samples)):
    print(f"Sample {i}: {samples[i]}")

In [ ]:
# enumerate() — get index AND value (preferred over range + index)
for i, sample in enumerate(samples):
    print(f"[{i}] {sample}")

---
## 3. `while` loops

A `while` loop runs as long as a condition is `True`. Use it when you don't know in advance how many iterations you need.

In [ ]:
# Count up to a threshold
reads = 0
target = 1000

while reads < target:
    reads += 100   # simulate adding 100 reads per cycle

print(f"Reached {reads} reads")

> **Tip:** In data analysis, `for` loops are far more common than `while` loops. Prefer `for` when you know the collection you're iterating over. Use `while` for convergence checks, retry logic, or reading streams.

---
## 4. List comprehensions — concise loops

A **list comprehension** builds a new list from an existing one in a single readable line. Very common in bioinformatics code.

In [ ]:
genes = ["BRCA1", "TP53", "EGFR", "MYC", "PTEN", "BCL2", "BRAF"]

# Get all genes starting with 'B'
b_genes = [g for g in genes if g.startswith("B")]
print("Genes starting with B:", b_genes)

# Get lengths of all gene names
lengths = [len(g) for g in genes]
print("Name lengths:", lengths)

---
## 5. Defining functions

A **function** is a named, reusable block of code. Once defined, you can call it as many times as needed.

**Why write functions?**
- Avoid copy-pasting the same logic repeatedly
- Make your code readable — a well-named function documents itself
- Make testing and debugging easier

In [ ]:
# Basic function structure
def greet_gene(gene_name):
    """Print a greeting for a gene."""
    print(f"Hello, {gene_name}!")

# Call the function
greet_gene("BRCA1")
greet_gene("TP53")

In [ ]:
# Functions that return a value
def add_counts(a, t, g, c):
    """Return the total number of bases."""
    return a + t + g + c

total = add_counts(18, 12, 20, 10)
print("Total bases:", total)

In [ ]:
# Default parameter values
def classify_pvalue(p, threshold=0.05):
    """Return 'significant' or 'not significant' based on threshold."""
    if p < threshold:
        return "significant"
    else:
        return "not significant"

print(classify_pvalue(0.03))          # uses default threshold 0.05
print(classify_pvalue(0.03, 0.01))    # override threshold to 0.01

---
## 6. Worked example: `count_nucleotides(seq)`

Let's write a function that takes a DNA sequence and returns the count of each nucleotide.

In [ ]:
def count_nucleotides(seq):
    """
    Count occurrences of each nucleotide in a DNA sequence.
    
    Parameters
    ----------
    seq : str
        DNA sequence (case-insensitive)
    
    Returns
    -------
    dict
        Dictionary with keys 'A', 'T', 'G', 'C' and their counts.
    """
    seq = seq.upper()   # handle lowercase input
    counts = {
        "A": seq.count("A"),
        "T": seq.count("T"),
        "G": seq.count("G"),
        "C": seq.count("C")
    }
    return counts

# Test it
result = count_nucleotides("ATGCGATCGATCGAAC")
print(result)

# Access individual counts
print(f"Adenine: {result['A']}")

---
## 7. Worked example: `gc_content(seq)`

Now let's build on `count_nucleotides` to compute GC content. **Reusing functions keeps code DRY** (Don't Repeat Yourself).

In [ ]:
def gc_content(seq):
    """
    Compute the GC content of a DNA sequence.
    
    Returns
    -------
    float
        GC content as a percentage (0-100).
    """
    counts = count_nucleotides(seq)   # reuse our previous function!
    total = sum(counts.values())
    if total == 0:
        return 0.0
    gc = counts["G"] + counts["C"]
    return (gc / total) * 100

# Test on a few sequences
sequences = [
    ("ATATATAT", "AT-rich"),
    ("GCGCGCGC", "GC-rich"),
    ("ATGCATGC", "balanced"),
]

for seq, label in sequences:
    gc = gc_content(seq)
    print(f"{label:10s} | {seq} | GC = {gc:.1f}%")

---
## 8. Variable scope

Variables defined **inside** a function are *local* — they only exist within that function and disappear when it returns. This prevents accidental interference between different parts of your code.

In [ ]:
x = 100   # global variable

def my_function():
    x = 42   # local variable — does NOT affect the global x
    print("Inside function, x =", x)

my_function()
print("Outside function, x =", x)   # still 100

---
## Exercises

### Exercise 1
Write a function `complement(seq)` that returns the DNA complement of a sequence.

Rules: A↔T, G↔C (e.g., `"ATGC"` → `"TACG"`)

*Hint: iterate over each base and build the complement string character by character, using an `if/elif` chain.*

In [ ]:
# Exercise 1 — your code here
def complement(seq):
    pass   # replace this

# Test your function:
# complement("ATGC") should return "TACG"
# complement("AAAAAA") should return "TTTTTT"
print(complement("ATGC"))
print(complement("AAAAAA"))

### Exercise 2
Write a function `classify_expression(log2fc, pvalue)` that returns:
- `"upregulated"` if log2fc > 1 AND pvalue < 0.05
- `"downregulated"` if log2fc < -1 AND pvalue < 0.05
- `"unchanged"` otherwise

Test it with several combinations of values.

In [ ]:
# Exercise 2 — your code here
def classify_expression(log2fc, pvalue):
    pass   # replace this

# Test cases:
print(classify_expression(2.5, 0.01))   # upregulated
print(classify_expression(-1.8, 0.03))  # downregulated
print(classify_expression(1.5, 0.1))    # unchanged (p not significant)
print(classify_expression(0.3, 0.001))  # unchanged (fc too small)

### Exercise 3
Given this list of gene names:
```python
genes = ["BRCA1", "TP53", "MYC", "EGFR", "PTEN", "AKT1", "CDK4", "BCL2", "RB1", "KRAS"]
```
Use a loop or list comprehension to build a new list containing only the genes whose **name is longer than 4 characters**.

Print the result.

In [ ]:
# Exercise 3 — your code here
genes = ["BRCA1", "TP53", "MYC", "EGFR", "PTEN", "AKT1", "CDK4", "BCL2", "RB1", "KRAS"]


---
## Summary

| Concept | Syntax | Use case |
|---------|--------|----------|
| if/elif/else | `if p < 0.05:` | Classify results |
| for loop | `for gene in genes:` | Process each item |
| while loop | `while reads < 1000:` | Until convergence |
| list comprehension | `[g for g in genes if ...]` | Filter/transform lists |
| function | `def gc_content(seq):` | Reusable logic |

**Next session:** data structures (lists, dicts, sets) and file I/O — reading and writing biological data files.